This notebook extends the ["AI Agent (Part 2)"](./AI%20Agent%20%28Part%202%29.ipynb) example.

In [ ]:
# Install LangChain and OpenAI integration
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import Image
from google.colab import userdata
from langchain.agents import AgentState
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.runnables import Runnable, RunnableConfig  # RunnableConfig carries per-run settings (e.g. thread_id)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.base import BaseCheckpointSaver  # Abstract base class for all checkpoint backends
from langgraph.checkpoint.memory import InMemorySaver       # Stores checkpoints in RAM (suitable for demos/tests)
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import ToolNode
from pathlib import Path
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

def display_graph(runnable: Runnable, output_png: Path):
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

# Lists all saved checkpoints for a given thread config.
# A checkpoint is a snapshot of the full graph state saved after EACH node execution.
# This lets you audit every step or even replay/resume the graph from any point.
def explore_checkpoints(checkpointer: BaseCheckpointSaver, config: RunnableConfig):
    checkpoints = list(checkpointer.list(config))
    print(f"There are {len(checkpoints)} checkpoints in total:")
    for checkpoint in reversed(checkpoints):
        print(checkpoint)

# Walks through the full execution history of a graph run step by step.
# Each 'snapshot' shows: step number, state values at that step, and what node runs next.
def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

In [ ]:
# Same tools as AI Agent notebooks — fake weather and mini-encyclopedia search
@tool
def weather(city: str) -> str:
    """Return a (fake) current-weather report for a city."""
    data = {
        "sofia": "Sofia: 18 C, partly cloudy",
        "london": "London: 11 C, rainy",
        "tokyo": "Tokyo: 22 C, sunny",
    }
    return data.get(city.lower(), f"No data for {city}.")

@tool
def search(query: str) -> str:
    """Look up a term in the built-in mini-encyclopedia."""
    data = {
        "langgraph": "LangGraph is a library for building stateful, cyclic LLM apps.",
        "react": "ReAct is a prompting pattern: Reason then Act, in a loop.",
        "dag": "A DAG is a directed acyclic graph — no cycles allowed.",
    }
    return data.get(query.lower(), "(nothing found)")


SYSTEM_PROMPT = "You are a concise assistant. Use tools when useful."
TOOLS = [weather, search]

In [ ]:
model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(TOOLS)

# InMemorySaver stores checkpoints in RAM.
# For production, swap this with SqliteSaver or PostgresSaver for durable persistence.
checkpointer = InMemorySaver()

# Same extended state as AI Agent Part 2 — adds a model call counter
class CustomAgentState(AgentState):
    model_calls: int

# Lifecycle hooks — for logging/observability (same pattern as AI Agent Part 2)
def on_start(state: CustomAgentState):
    # Fires once at the very beginning of each graph invocation
    print("Our graph execution started!")

def on_end(state: CustomAgentState):
    # Fires once just before the graph exits
    print("Our graph execution just ended!")

def before_model_node(state: CustomAgentState):
    # Runs immediately before each model call — increments and logs the counter
    model_call_id = state.get('model_calls', 0) + 1
    print('=' * 20)
    print(f"Starting model call #{model_call_id}...")
    return { "model_calls": model_call_id }

def after_model_node(state: CustomAgentState):
    # Runs immediately after each model call — logs the completed call number
    model_call_id = state.get('model_calls', 0)
    print(f"Finished model call #{model_call_id}.")
    print('=' * 20)

def model_node(state: CustomAgentState):
    # The actual GPT call with the full conversation history
    response = model.invoke([SystemMessage(SYSTEM_PROMPT), *state["messages"]])
    return { "messages": [response] }

In [ ]:
# Same routing function as previous notebooks — checks for pending tool calls
def has_pending_tool_calls(state: CustomAgentState) -> bool:
    messages = state.get("messages", [])
    if not messages:
        return False

    last_message = messages[-1]
    return isinstance(last_message, AIMessage) and last_message.tool_calls

In [ ]:
in_memory_checkpointer = InMemorySaver()

In [ ]:
# Build the same agent graph as Part 2, but THIS TIME pass the checkpointer when compiling.
# The checkpointer is what gives the graph "memory" across multiple .invoke() calls —
# each invocation with the same thread_id will remember the previous conversation.
graph_builder = StateGraph(CustomAgentState)
graph_builder.add_node("on_start", on_start)
graph_builder.add_node("on_end", on_end)
graph_builder.add_node("before_model", before_model_node)
graph_builder.add_node("after_model", after_model_node)
graph_builder.add_node("model", model_node)
graph_builder.add_node("tools", ToolNode(TOOLS))

graph_builder.add_edge(START, "on_start")
graph_builder.add_edge("on_start", "before_model")
graph_builder.add_edge("before_model", "model")
graph_builder.add_edge("model", "after_model")
graph_builder.add_conditional_edges("after_model", lambda x: "tools" if has_pending_tool_calls(x) else "on_end", ["tools", "on_end"])
graph_builder.add_edge("tools", "before_model")
graph_builder.add_edge("on_end", END)

# KEY DIFFERENCE from AI Agent Part 2: checkpointer=checkpointer enables persistent memory
graph = graph_builder.compile(checkpointer=in_memory_checkpointer)

In [ ]:
# Visualize the graph structure (same as AI Agent Part 2)
display_graph(graph, Path("/content/graph.png"))

In [ ]:
# A 'thread_id' identifies a unique conversation session.
# All invocations sharing the same thread_id will be linked together,
# so the model remembers earlier messages (like a real chat session).
thread1_config = { "configurable": { "thread_id": "thread_1" } }

# First message in thread_17
thread1_result = graph.invoke(
    input={
        "messages": [HumanMessage("What's the weather in Tokyo and what is LangGraph, briefly?")]
    },
    config=thread1_config  # Links this invocation to thread_17
)

In [ ]:
# Print the conversation from the first run
print_conversation(thread1_result["messages"])

In [ ]:
# Inspect how many checkpoints were saved during the first run.
# Every node execution saves a checkpoint — so you should see several here.
explore_checkpoints(in_memory_checkpointer, thread1_config)

In [ ]:
# Walk through the full state evolution step by step for the first run
explore_state_history(graph, thread1_config)

In [ ]:
# Send a FOLLOW-UP message in the SAME thread (thread_17).
# Because we use the same thread_id, the checkpointer loads the previous conversation history.
# The model will understand "And what about London?" as a weather follow-up question.
thread1_result = graph.invoke(
    input={
        "messages": [HumanMessage("And what about London?")]
    },
    config=thread1_config  # Same thread — conversation history is preserved automatically
)

In [ ]:
# Print the full conversation — it will show ALL messages from both runs combined
print_conversation(thread1_result["messages"])

In [ ]:
# Check checkpoints again — there will be MORE now after the second run
explore_checkpoints(in_memory_checkpointer, thread1_config)

In [ ]:
# Walk through the FULL state history covering both the first and second runs
explore_state_history(graph, thread1_config)